# 🛒 SentimentIQ — MockShop Review Scraper + Sentiment Analyzer

**School Project** | Text Classification & Sentiment Analysis Pipeline

This notebook scrapes product reviews from the **MockShop demo page** (`mockshop_product.html`)  
and runs full sentiment analysis using VADER + TextBlob — no API key, no internet needed.

---
### Pipeline
```
mockshop_product.html
        │
        ▼
  BeautifulSoup Scraper   ← parses review text, rating, author, date
        │
        ▼
  VADER + TextBlob        ← ensemble sentiment scoring
        │
        ▼
  CSV / JSON / HTML       ← reports & interactive dashboard
```

### Setup
1. Make sure `mockshop_product.html` is in the **same folder** as this notebook
2. Run all cells top to bottom

---

## Cell 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4 lxml vaderSentiment textblob pandas matplotlib -q

import nltk
nltk.download('punkt',        quiet=True)
nltk.download('punkt_tab',    quiet=True)
nltk.download('averaged_perceptron_tagger',     quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print('✓ All packages ready')


## Cell 2 — Initialize Variables
Safe defaults so no cell ever crashes with `NameError`.

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd

reviews    = []    # raw scraped reviews
results    = []    # sentiment results
agg        = {}    # aggregate stats
html_path  = None  # saved report path
results_df = None

OUTPUT_DIR = Path('./mockshop_output')
OUTPUT_DIR.mkdir(exist_ok=True)
TIMESTAMP  = datetime.now().strftime('%Y%m%d_%H%M%S')

# Path to the mock HTML file — change if it's in a different folder
HTML_FILE  = Path('mockshop_product.html')

print(f'Output folder : {OUTPUT_DIR.resolve()}')
print(f'Mock page     : {HTML_FILE.resolve()}')
print(f'File exists   : {HTML_FILE.exists()}')


## Cell 3 — Sentiment Analyzer Engine
VADER + TextBlob ensemble with category, intent, keyword, and dimension scoring.

In [ ]:
"""
Sentiment Analysis Engine
--------------------------
Ensemble of VADER (rule-based, great for social/review text) +
TextBlob (pattern-based NLP) for robust sentiment scoring
without needing an external API.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Literal

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# ── Category keywords ──────────────────────────────────────────────────────────
CATEGORY_RULES: dict[str, list[str]] = {
    "Product Quality":     ["quality", "build", "material", "durable", "broken", "defective",
                            "well made", "cheap", "sturdy", "flimsy", "excellent", "poor quality"],
    "Shipping & Delivery": ["shipping", "delivery", "arrived", "package", "delayed", "late",
                            "fast ship", "slow ship", "tracking", "courier", "box", "damaged in transit"],
    "Customer Service":    ["customer service", "support", "representative", "agent", "response",
                            "helpful", "rude", "ignored", "refund request", "contact", "resolved"],
    "Pricing & Value":     ["price", "value", "expensive", "cheap", "worth", "cost", "affordable",
                            "overpriced", "bargain", "deal", "money"],
    "Packaging":           ["packaging", "pack", "unboxing", "wrapped", "box", "protective",
                            "bag", "envelope", "presentation"],
    "Authenticity":        ["fake", "counterfeit", "genuine", "original", "authentic", "knockoff",
                            "not original", "replica", "as described"],
    "Returns & Refunds":   ["return", "refund", "exchange", "sent back", "return policy",
                            "money back", "replacement", "warranty"],
    "Ease of Use":         ["easy", "simple", "intuitive", "complicated", "difficult",
                            "instructions", "setup", "install", "user friendly"],
}

# ── Intent rules ───────────────────────────────────────────────────────────────
INTENT_RULES: dict[str, list[str]] = {
    "Churn Risk":          ["never again", "won't buy", "will not return", "last time",
                            "switching", "competitor", "avoid", "do not buy"],
    "Escalation Required": ["lawsuit", "report", "bbb", "attorney general", "fraud",
                            "scam", "legal action", "police"],
    "Loyal Customer":      ["always buy", "repeat customer", "love this brand", "5 stars again",
                            "been buying for years", "my go-to"],
    "Feature Request":     ["should have", "wish it had", "would be better if", "please add",
                            "suggest", "improvement", "next version"],
    "Support Needed":      ["help", "how do i", "not working", "broken", "contact",
                            "can someone", "please fix"],
    "Praise":              ["love", "amazing", "fantastic", "best purchase", "highly recommend",
                            "exceeded expectations", "perfect", "wonderful"],
    "General Complaint":   ["disappointed", "unhappy", "terrible", "awful", "horrible",
                            "waste of money", "regret"],
}


@dataclass
class SentimentResult:
    # Core sentiment
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float          # 0–100
    label_emoji: str

    # Scores
    vader_compound: float      # -1 to 1
    vader_pos: float
    vader_neg: float
    vader_neu: float
    textblob_polarity: float   # -1 to 1
    textblob_subjectivity: float  # 0 to 1
    ensemble_score: float      # -1 to 1 (weighted average)

    # Dimensions (0–100)
    urgency: int
    frustration: int
    satisfaction: int
    loyalty_risk: int

    # Classification
    categories: list[str]
    intents: list[str]
    keywords: list[str]

    # Original
    rating: float              # star rating from Amazon
    review_title: str
    review_body: str

    def to_dict(self) -> dict:
        return {
            "sentiment": self.sentiment,
            "confidence": self.confidence,
            "label_emoji": self.label_emoji,
            "vader_compound": round(self.vader_compound, 4),
            "textblob_polarity": round(self.textblob_polarity, 4),
            "textblob_subjectivity": round(self.textblob_subjectivity, 4),
            "ensemble_score": round(self.ensemble_score, 4),
            "urgency": self.urgency,
            "frustration": self.frustration,
            "satisfaction": self.satisfaction,
            "loyalty_risk": self.loyalty_risk,
            "categories": self.categories,
            "intents": self.intents,
            "keywords": self.keywords,
            "star_rating": self.rating,
            "review_title": self.review_title,
            "review_body": self.review_body,
        }


_vader = SentimentIntensityAnalyzer()


def _extract_categories(text: str) -> list[str]:
    text_lower = text.lower()
    found = []
    for cat, keywords in CATEGORY_RULES.items():
        if any(kw in text_lower for kw in keywords):
            found.append(cat)
    return found or ["General Feedback"]


def _extract_intents(text: str) -> list[str]:
    text_lower = text.lower()
    found = []
    for intent, keywords in INTENT_RULES.items():
        if any(kw in text_lower for kw in keywords):
            found.append(intent)
    return found or ["Feedback Only"]


def _extract_keywords(text: str, top_n: int = 8) -> list[str]:
    """Simple frequency-based keyword extraction, filtering stopwords."""
    STOPWORDS = {
        "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
        "of", "with", "by", "from", "is", "it", "this", "that", "was", "are",
        "be", "have", "had", "has", "not", "i", "my", "we", "our", "they",
        "their", "you", "your", "so", "very", "just", "also", "as", "if",
        "its", "than", "then", "when", "what", "would", "could", "should",
        "will", "did", "do", "does", "get", "got", "been", "me", "him",
        "her", "us", "no", "up", "out", "about", "after", "before", "more",
        "all", "any", "one", "can", "there", "which", "who", "how", "some",
    }
    words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
    freq: dict[str, int] = {}
    for w in words:
        if w not in STOPWORDS:
            freq[w] = freq.get(w, 0) + 1
    return sorted(freq, key=freq.get, reverse=True)[:top_n]  # type: ignore


def _compute_dimensions(text: str, vader_scores: dict, rating: float) -> dict:
    text_lower = text.lower()

    urgency_signals = ["immediately", "asap", "urgent", "right now", "right away",
                       "still waiting", "weeks", "unacceptable", "lawsuit", "!!"]
    urgency = min(100, sum(10 for s in urgency_signals if s in text_lower))
    if rating <= 1:
        urgency = min(100, urgency + 20)

    frustration = int(max(0, min(100, (-vader_scores["compound"] + 1) / 2 * 100)))
    satisfaction = int(max(0, min(100, (vader_scores["compound"] + 1) / 2 * 100)))

    loyalty_signals = ["never again", "won't buy", "avoid", "never return", "last time",
                       "do not buy", "scam", "fraud", "one star"]
    loyalty_risk = min(100, sum(20 for s in loyalty_signals if s in text_lower))
    if rating <= 2:
        loyalty_risk = min(100, loyalty_risk + 30)

    return {
        "urgency": urgency,
        "frustration": frustration,
        "satisfaction": satisfaction,
        "loyalty_risk": loyalty_risk,
    }


def analyze(review_title: str, review_body: str, rating: float = 0.0) -> SentimentResult:
    """
    Run ensemble sentiment analysis on a single review.
    """
    full_text = f"{review_title}. {review_body}".strip()

    # ── VADER ──────────────────────────────────────────────────────────────────
    vader_scores = _vader.polarity_scores(full_text)
    vc = vader_scores["compound"]

    # ── TextBlob ───────────────────────────────────────────────────────────────
    blob = TextBlob(full_text)
    tb_pol = blob.sentiment.polarity
    tb_sub = blob.sentiment.subjectivity

    # ── Ensemble: 50% VADER + 30% TextBlob + 20% star rating signal ───────────
    star_signal = 0.0
    if rating > 0:
        star_signal = (rating - 3.0) / 2.0  # maps 1–5 → -1 to 1

    ensemble = (0.50 * vc) + (0.30 * tb_pol) + (0.20 * star_signal)
    ensemble = max(-1.0, min(1.0, ensemble))

    # ── Label ─────────────────────────────────────────────────────────────────
    # "mixed" = VADER and TextBlob disagree in sign significantly
    disagree = (vc > 0.2 and tb_pol < -0.1) or (vc < -0.2 and tb_pol > 0.1)

    if disagree:
        sentiment = "mixed"
        label_emoji = "🎭"
        confidence = round(50 + abs(ensemble) * 30)
    elif ensemble >= 0.15:
        sentiment = "positive"
        label_emoji = "😊"
        confidence = round(min(99, 50 + ensemble * 50))
    elif ensemble <= -0.15:
        sentiment = "negative"
        label_emoji = "😠"
        confidence = round(min(99, 50 + abs(ensemble) * 50))
    else:
        sentiment = "neutral"
        label_emoji = "😐"
        confidence = round(60 + (0.15 - abs(ensemble)) / 0.15 * 20)

    dims = _compute_dimensions(full_text, vader_scores, rating)
    categories = _extract_categories(full_text)
    intents = _extract_intents(full_text)
    keywords = _extract_keywords(full_text)

    return SentimentResult(
        sentiment=sentiment,
        confidence=confidence,
        label_emoji=label_emoji,
        vader_compound=vc,
        vader_pos=vader_scores["pos"],
        vader_neg=vader_scores["neg"],
        vader_neu=vader_scores["neu"],
        textblob_polarity=tb_pol,
        textblob_subjectivity=tb_sub,
        ensemble_score=ensemble,
        urgency=dims["urgency"],
        frustration=dims["frustration"],
        satisfaction=dims["satisfaction"],
        loyalty_risk=dims["loyalty_risk"],
        categories=categories,
        intents=intents,
        keywords=keywords,
        rating=rating,
        review_title=review_title,
        review_body=review_body,
    )


def analyze_batch(reviews: list) -> list[SentimentResult]:
    """Analyze a list of Review objects from scraper.py."""
    return [analyze(r.title, r.body, r.rating) for r in reviews]

print('✓ Analyzer ready')

## Cell 4 — Reporter (CSV / JSON / HTML Dashboard)

In [ ]:
"""
Report Generator
-----------------
Produces CSV, JSON, and a self-contained HTML dashboard
from a list of SentimentResult objects.
"""

from __future__ import annotations

import json
import csv
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import List



# ─────────────────────────────────────────────────────────────────────────────
# Aggregate stats
# ─────────────────────────────────────────────────────────────────────────────

def aggregate(results: List[SentimentResult]) -> dict:
    total = len(results)
    if total == 0:
        return {}

    sentiment_counts = Counter(r.sentiment for r in results)
    category_counts = Counter(c for r in results for c in r.categories)
    intent_counts = Counter(i for r in results for i in r.intents)
    keyword_counts = Counter(k for r in results for k in r.keywords)

    avg_ensemble = sum(r.ensemble_score for r in results) / total
    avg_confidence = sum(r.confidence for r in results) / total
    avg_rating = sum(r.rating for r in results if r.rating > 0)
    rated = [r for r in results if r.rating > 0]
    avg_rating = sum(r.rating for r in rated) / len(rated) if rated else 0.0

    avg_urgency = sum(r.urgency for r in results) / total
    avg_frustration = sum(r.frustration for r in results) / total
    avg_satisfaction = sum(r.satisfaction for r in results) / total
    avg_loyalty_risk = sum(r.loyalty_risk for r in results) / total

    return {
        "total_reviews": total,
        "sentiment_distribution": dict(sentiment_counts),
        "dominant_sentiment": sentiment_counts.most_common(1)[0][0],
        "avg_ensemble_score": round(avg_ensemble, 4),
        "avg_confidence": round(avg_confidence, 1),
        "avg_star_rating": round(avg_rating, 2),
        "top_categories": dict(category_counts.most_common(6)),
        "top_intents": dict(intent_counts.most_common(5)),
        "top_keywords": dict(keyword_counts.most_common(10)),
        "avg_dimensions": {
            "urgency": round(avg_urgency, 1),
            "frustration": round(avg_frustration, 1),
            "satisfaction": round(avg_satisfaction, 1),
            "loyalty_risk": round(avg_loyalty_risk, 1),
        },
        "churn_risk_reviews": sum(1 for r in results if "Churn Risk" in r.intents),
        "escalation_needed": sum(1 for r in results if "Escalation Required" in r.intents),
    }


# ─────────────────────────────────────────────────────────────────────────────
# CSV
# ─────────────────────────────────────────────────────────────────────────────

def save_csv(results: List[SentimentResult], path: str | Path) -> Path:
    path = Path(path)
    fieldnames = [
        "sentiment", "confidence", "star_rating", "ensemble_score",
        "vader_compound", "textblob_polarity", "textblob_subjectivity",
        "urgency", "frustration", "satisfaction", "loyalty_risk",
        "categories", "intents", "keywords",
        "review_title", "review_body",
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            row = r.to_dict()
            row["categories"] = "; ".join(row["categories"])
            row["intents"] = "; ".join(row["intents"])
            row["keywords"] = "; ".join(row["keywords"])
            writer.writerow({k: row[k] for k in fieldnames if k in row})
    return path


# ─────────────────────────────────────────────────────────────────────────────
# JSON
# ─────────────────────────────────────────────────────────────────────────────

def save_json(results: List[SentimentResult], agg: dict, path: str | Path) -> Path:
    path = Path(path)
    data = {
        "generated_at": datetime.now().isoformat(),
        "aggregate": agg,
        "reviews": [r.to_dict() for r in results],
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# HTML Dashboard
# ─────────────────────────────────────────────────────────────────────────────

def save_html(results: List[SentimentResult], agg: dict, path: str | Path, product_url: str = "") -> Path:
    path = Path(path)

    # Prepare data for JS
    reviews_json = json.dumps([r.to_dict() for r in results], ensure_ascii=False)
    agg_json = json.dumps(agg, ensure_ascii=False)

    sentiment_colors = {
        "positive": "#4ade80",
        "negative": "#f87171",
        "neutral":  "#94a3b8",
        "mixed":    "#fbbf24",
    }

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>SentimentIQ — Amazon Review Report</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Syne:wght@400;600;700;800&display=swap');
:root {{
  --bg:#0a0a0f;--surface:#12121a;--surface2:#1a1a26;--border:#2a2a3d;
  --accent:#7c6af7;--accent2:#f76a8c;--accent3:#6af7c8;
  --text:#e8e8f0;--muted:#6b6b8a;
  --positive:#4ade80;--negative:#f87171;--neutral:#94a3b8;--mixed:#fbbf24;
}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'Syne',sans-serif;background:var(--bg);color:var(--text);min-height:100vh}}
body::before{{content:'';position:fixed;inset:0;
  background:radial-gradient(ellipse 60% 40% at 20% 10%,rgba(124,106,247,.10) 0%,transparent 60%),
             radial-gradient(ellipse 40% 60% at 80% 90%,rgba(247,106,140,.07) 0%,transparent 60%);
  pointer-events:none;z-index:0}}
.wrap{{max-width:1200px;margin:0 auto;padding:40px 24px;position:relative;z-index:1}}
header{{margin-bottom:40px}}
.logo{{font-size:30px;font-weight:800;letter-spacing:-1.5px;
  background:linear-gradient(135deg,var(--accent),var(--accent2));
  -webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text}}
.sub{{font-family:'DM Mono',monospace;font-size:11px;color:var(--muted);letter-spacing:2px;margin-top:4px}}
.product-link{{font-family:'DM Mono',monospace;font-size:12px;color:var(--accent);margin-top:8px;word-break:break-all}}
/* KPI row */
.kpi-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:16px;margin-bottom:28px}}
.kpi{{background:var(--surface);border:1px solid var(--border);border-radius:14px;padding:20px 18px}}
.kpi-label{{font-family:'DM Mono',monospace;font-size:10px;color:var(--muted);letter-spacing:1.5px;text-transform:uppercase;margin-bottom:10px}}
.kpi-val{{font-size:32px;font-weight:800;letter-spacing:-2px}}
.kpi-sub{{font-family:'DM Mono',monospace;font-size:11px;color:var(--muted);margin-top:4px}}
/* Grid */
.grid2{{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:20px}}
@media(max-width:700px){{.grid2{{grid-template-columns:1fr}}}}
.card{{background:var(--surface);border:1px solid var(--border);border-radius:14px;padding:22px}}
.card-title{{font-family:'DM Mono',monospace;font-size:10px;letter-spacing:2px;text-transform:uppercase;
  color:var(--muted);margin-bottom:18px;display:flex;align-items:center;gap:8px}}
.card-title::after{{content:'';flex:1;height:1px;background:var(--border)}}
canvas{{width:100%!important}}
/* Sentiment bar */
.sent-bar{{display:flex;height:28px;border-radius:8px;overflow:hidden;margin-bottom:18px}}
.sent-seg{{display:flex;align-items:center;justify-content:center;font-family:'DM Mono',monospace;
  font-size:11px;font-weight:500;transition:flex .5s ease;overflow:hidden;white-space:nowrap}}
/* Dimension bars */
.dim-item{{margin-bottom:14px}}
.dim-header{{display:flex;justify-content:space-between;margin-bottom:6px;font-family:'DM Mono',monospace;font-size:12px}}
.dim-track{{height:6px;background:var(--border);border-radius:100px;overflow:hidden}}
.dim-fill{{height:100%;border-radius:100px}}
/* Tags */
.tags{{display:flex;flex-wrap:wrap;gap:8px}}
.tag{{padding:5px 12px;border-radius:100px;font-family:'DM Mono',monospace;font-size:11px;
  border:1px solid;display:inline-block}}
.tag.cat{{background:rgba(124,106,247,.1);border-color:rgba(124,106,247,.3);color:var(--accent)}}
.tag.kw{{background:rgba(106,247,200,.08);border-color:rgba(106,247,200,.25);color:var(--accent3)}}
.tag.intent{{background:rgba(247,106,140,.08);border-color:rgba(247,106,140,.25);color:var(--accent2)}}
/* Alert boxes */
.alert-row{{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px}}
.alert-box{{border-radius:12px;padding:16px 18px;border:1px solid}}
.alert-box.danger{{background:rgba(248,113,113,.08);border-color:rgba(248,113,113,.3)}}
.alert-box.warn{{background:rgba(251,191,36,.08);border-color:rgba(251,191,36,.3)}}
.alert-val{{font-size:28px;font-weight:800;margin-bottom:4px}}
.alert-box.danger .alert-val{{color:var(--negative)}}
.alert-box.warn .alert-val{{color:var(--mixed)}}
.alert-label{{font-family:'DM Mono',monospace;font-size:11px;color:var(--muted)}}
/* Table */
.table-wrap{{overflow-x:auto;margin-bottom:20px}}
table{{width:100%;border-collapse:collapse;font-size:13px}}
th{{font-family:'DM Mono',monospace;font-size:10px;letter-spacing:1.5px;text-transform:uppercase;
  color:var(--muted);text-align:left;padding:10px 12px;border-bottom:1px solid var(--border)}}
td{{padding:10px 12px;border-bottom:1px solid rgba(42,42,61,.5);vertical-align:top;max-width:320px}}
tr:hover td{{background:var(--surface2)}}
.sent-pill{{display:inline-block;padding:3px 10px;border-radius:100px;font-family:'DM Mono',monospace;
  font-size:10px;font-weight:500;border:1px solid}}
.sent-pill.positive{{background:rgba(74,222,128,.1);border-color:rgba(74,222,128,.3);color:var(--positive)}}
.sent-pill.negative{{background:rgba(248,113,113,.1);border-color:rgba(248,113,113,.3);color:var(--negative)}}
.sent-pill.neutral{{background:rgba(148,163,184,.1);border-color:rgba(148,163,184,.3);color:var(--neutral)}}
.sent-pill.mixed{{background:rgba(251,191,36,.1);border-color:rgba(251,191,36,.3);color:var(--mixed)}}
.stars{{color:#fbbf24;font-size:13px}}
/* Filter bar */
.filter-bar{{display:flex;gap:10px;margin-bottom:16px;flex-wrap:wrap}}
.filter-btn{{font-family:'DM Mono',monospace;font-size:11px;padding:6px 14px;border-radius:8px;
  border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;transition:all .2s}}
.filter-btn.active{{border-color:var(--accent);color:var(--accent);background:rgba(124,106,247,.1)}}
.filter-btn:hover:not(.active){{border-color:var(--muted)}}
/* footer */
footer{{text-align:center;font-family:'DM Mono',monospace;font-size:11px;color:var(--muted);margin-top:40px;padding-top:20px;border-top:1px solid var(--border)}}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <div class="logo">SentimentIQ</div>
    <div class="sub">AMAZON REVIEW SENTIMENT REPORT &nbsp;·&nbsp; Generated {datetime.now().strftime("%Y-%m-%d %H:%M")}</div>
    {"<div class='product-link'>🔗 " + product_url + "</div>" if product_url else ""}
  </header>

  <!-- KPIs -->
  <div class="kpi-row" id="kpiRow"></div>

  <!-- Alerts -->
  <div class="alert-row" id="alertRow"></div>

  <!-- Charts row -->
  <div class="grid2">
    <div class="card">
      <div class="card-title">Sentiment Distribution</div>
      <div class="sent-bar" id="sentBar"></div>
      <canvas id="donutChart" height="220"></canvas>
    </div>
    <div class="card">
      <div class="card-title">Average Dimensions</div>
      <div id="dimBars"></div>
    </div>
  </div>

  <div class="grid2">
    <div class="card">
      <div class="card-title">Top Categories</div>
      <canvas id="catChart" height="220"></canvas>
    </div>
    <div class="card">
      <div class="card-title">Top Keywords</div>
      <div class="tags" id="keywordCloud"></div>
    </div>
  </div>

  <!-- Intents -->
  <div class="card" style="margin-bottom:20px">
    <div class="card-title">Detected Intents</div>
    <div class="tags" id="intentTags"></div>
  </div>

  <!-- Reviews table -->
  <div class="card">
    <div class="card-title">All Reviews</div>
    <div class="filter-bar" id="filterBar">
      <button class="filter-btn active" onclick="filterTable('all')">All</button>
      <button class="filter-btn" onclick="filterTable('positive')">😊 Positive</button>
      <button class="filter-btn" onclick="filterTable('negative')">😠 Negative</button>
      <button class="filter-btn" onclick="filterTable('neutral')">😐 Neutral</button>
      <button class="filter-btn" onclick="filterTable('mixed')">🎭 Mixed</button>
    </div>
    <div class="table-wrap">
      <table id="reviewTable">
        <thead>
          <tr>
            <th>Sentiment</th>
            <th>Stars</th>
            <th>Conf.</th>
            <th>Title</th>
            <th>Excerpt</th>
            <th>Categories</th>
            <th>Intent</th>
          </tr>
        </thead>
        <tbody id="tableBody"></tbody>
      </table>
    </div>
  </div>

  <footer>SentimentIQ · Python VADER + TextBlob ensemble · No AI API required</footer>
</div>

<script>
const DATA = {reviews_json};
const AGG = {agg_json};
const SENT_COLORS = {json.dumps(sentiment_colors)};

// ── KPIs ────────────────────────────────────────────────────────────────────
const dist = AGG.sentiment_distribution || {{}};
const total = AGG.total_reviews || 0;

const kpis = [
  {{ label:'Total Reviews', val: total, sub:'scraped & analyzed' }},
  {{ label:'Dominant Sentiment', val: (AGG.dominant_sentiment||'—').toUpperCase(), sub:'overall mood' }},
  {{ label:'Avg Star Rating', val: AGG.avg_star_rating ? AGG.avg_star_rating + ' ★' : '—', sub:'out of 5.0' }},
  {{ label:'Avg Confidence', val: (AGG.avg_confidence||0) + '%', sub:'classifier certainty' }},
  {{ label:'Positive', val: dist.positive||0, sub:pct(dist.positive,total) }},
  {{ label:'Negative', val: dist.negative||0, sub:pct(dist.negative,total) }},
];

function pct(n,t){{ return t ? Math.round((n||0)/t*100)+'% of total' : '' }}

const kpiRow = document.getElementById('kpiRow');
kpis.forEach(k => {{
  const c = document.createElement('div'); c.className='kpi';
  const color = k.label==='Positive'? 'var(--positive)' : k.label==='Negative'? 'var(--negative)' : 'var(--text)';
  c.innerHTML=`<div class="kpi-label">${{k.label}}</div><div class="kpi-val" style="color:${{color}}">${{k.val}}</div><div class="kpi-sub">${{k.sub}}</div>`;
  kpiRow.appendChild(c);
}});

// ── Alerts ───────────────────────────────────────────────────────────────────
document.getElementById('alertRow').innerHTML=`
  <div class="alert-box danger">
    <div class="alert-val">${{AGG.churn_risk_reviews||0}}</div>
    <div class="alert-label">⚠ CHURN RISK REVIEWS — Require immediate attention</div>
  </div>
  <div class="alert-box warn">
    <div class="alert-val">${{AGG.escalation_needed||0}}</div>
    <div class="alert-label">🔺 ESCALATION REQUIRED — Legal/fraud signals detected</div>
  </div>`;

// ── Sentiment bar ────────────────────────────────────────────────────────────
const sentBar = document.getElementById('sentBar');
['positive','negative','neutral','mixed'].forEach(s => {{
  const n = dist[s]||0; if(!n) return;
  const pct_ = Math.round(n/total*100);
  const seg = document.createElement('div'); seg.className='sent-seg';
  seg.style.cssText=`flex:${{pct_}};background:${{SENT_COLORS[s]}};color:#0a0a0f;`;
  seg.textContent = pct_>8 ? pct_+'%' : '';
  sentBar.appendChild(seg);
}});

// ── Donut chart ──────────────────────────────────────────────────────────────
new Chart(document.getElementById('donutChart'), {{
  type:'doughnut',
  data:{{
    labels: Object.keys(dist),
    datasets:[{{ data: Object.values(dist),
      backgroundColor: Object.keys(dist).map(s=>SENT_COLORS[s]||'#666'),
      borderWidth:0, hoverOffset:6 }}]
  }},
  options:{{ plugins:{{ legend:{{ labels:{{ color:'#e8e8f0', font:{{family:'DM Mono',size:11}} }} }} }}, cutout:'65%' }}
}});

// ── Dimension bars ───────────────────────────────────────────────────────────
const dims = AGG.avg_dimensions || {{}};
const dimColors = {{ urgency:'#f87171', frustration:'#fb923c', satisfaction:'#4ade80', loyalty_risk:'#818cf8' }};
const dimContainer = document.getElementById('dimBars');
Object.entries(dims).forEach(([k,v]) => {{
  dimContainer.innerHTML += `
    <div class="dim-item">
      <div class="dim-header"><span style="text-transform:capitalize">${{k.replace('_',' ')}}</span><span style="color:var(--muted)">${{v}}%</span></div>
      <div class="dim-track"><div class="dim-fill" style="width:${{v}}%;background:${{dimColors[k]||'var(--accent)'}};transition:width .8s ease"></div></div>
    </div>`;
}});

// ── Category chart ───────────────────────────────────────────────────────────
const cats = AGG.top_categories||{{}};
new Chart(document.getElementById('catChart'), {{
  type:'bar',
  data:{{
    labels: Object.keys(cats),
    datasets:[{{ data: Object.values(cats),
      backgroundColor:'rgba(124,106,247,0.7)', borderRadius:6, borderSkipped:false }}]
  }},
  options:{{
    indexAxis:'y', responsive:true,
    plugins:{{ legend:{{ display:false }} }},
    scales:{{
      x:{{ ticks:{{ color:'#6b6b8a',font:{{family:'DM Mono',size:10}} }}, grid:{{ color:'#2a2a3d' }} }},
      y:{{ ticks:{{ color:'#e8e8f0',font:{{family:'DM Mono',size:10}} }}, grid:{{ display:false }} }}
    }}
  }}
}});

// ── Keyword cloud ─────────────────────────────────────────────────────────────
const kws = AGG.top_keywords||{{}};
const maxKw = Math.max(...Object.values(kws));
const kwCloud = document.getElementById('keywordCloud');
Object.entries(kws).forEach(([k,v]) => {{
  const size = 11 + Math.round((v/maxKw)*8);
  kwCloud.innerHTML += `<span class="tag kw" style="font-size:${{size}}px">#${{k}} (${{v}})</span>`;
}});

// ── Intent tags ───────────────────────────────────────────────────────────────
const intents = AGG.top_intents||{{}};
const intentEl = document.getElementById('intentTags');
Object.entries(intents).forEach(([k,v]) => {{
  intentEl.innerHTML += `<span class="tag intent">↗ ${{k}} <span style="opacity:.6">(${{v}})</span></span>`;
}});

// ── Table ─────────────────────────────────────────────────────────────────────
let currentFilter = 'all';

function renderTable(filter) {{
  currentFilter = filter;
  const rows = filter==='all' ? DATA : DATA.filter(r=>r.sentiment===filter);
  const stars = n => n>0 ? '★'.repeat(Math.round(n))+'☆'.repeat(5-Math.round(n)) : '—';
  document.getElementById('tableBody').innerHTML = rows.map(r => `
    <tr class="review-row" data-sentiment="${{r.sentiment}}">
      <td><span class="sent-pill ${{r.sentiment}}">${{r.label_emoji}} ${{r.sentiment}}</span></td>
      <td><span class="stars">${{stars(r.star_rating)}}</span></td>
      <td style="font-family:'DM Mono',monospace;font-size:12px">${{r.confidence}}%</td>
      <td style="font-weight:600;min-width:140px">${{r.review_title||'—'}}</td>
      <td style="color:var(--muted);font-size:12px">${{(r.review_body||'').substring(0,120)}}...</td>
      <td>${{(r.categories||[]).map(c=>`<span class="tag cat" style="margin:2px">${{c}}</span>`).join('')}}</td>
      <td>${{(r.intents||[]).map(i=>`<span class="tag intent" style="margin:2px">${{i}}</span>`).join('')}}</td>
    </tr>`).join('');
}}

function filterTable(f) {{
  document.querySelectorAll('.filter-btn').forEach(b=>b.classList.remove('active'));
  event.target.classList.add('active');
  renderTable(f);
}}

renderTable('all');
</script>
</body>
</html>"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path

print('✓ Reporter ready')

## Cell 5 — 🕷️ Scrape Reviews from MockShop HTML
Uses BeautifulSoup to parse the local HTML file and extract every review: text, rating, author, date, variation.

In [ ]:
from bs4 import BeautifulSoup
from dataclasses import dataclass

@dataclass
class Review:
    text      : str
    rating    : int    # 1-5
    author    : str
    date      : str
    variation : str
    helpful   : int
    has_photo : bool

def scrape_mockshop(html_path: Path) -> list:
    """Parse all reviews from the MockShop HTML file."""
    with open(html_path, encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')

    scraped = []
    items = soup.select('.review-item')
    print(f'Found {len(items)} review containers')

    for item in items:
        try:
            # Rating from data attribute
            rating = int(item.get('data-rating', 0))

            # Author
            author_el = item.select_one('.reviewer-name')
            author = author_el.get_text(strip=True) if author_el else 'Anonymous'

            # Review text
            text_el = item.select_one('.review-text')
            text = text_el.get_text(strip=True) if text_el else ''

            # Date
            date_el = item.select_one('.review-date')
            date = date_el.get_text(strip=True) if date_el else ''

            # Variation
            var_el = item.select_one('.review-variation')
            variation = var_el.get_text(strip=True).replace('Variation:', '').strip() if var_el else ''

            # Helpful count
            helpful_el = item.select_one('.helpful-btn')
            helpful = 0
            if helpful_el:
                import re
                m = re.search(r'(\d+)', helpful_el.get_text())
                helpful = int(m.group(1)) if m else 0

            # Has photo
            has_photo = bool(item.select('.review-img'))

            if text:
                scraped.append(Review(
                    text=text, rating=rating, author=author,
                    date=date, variation=variation,
                    helpful=helpful, has_photo=has_photo
                ))
        except Exception as e:
            print(f'  Skipped item: {e}')
            continue

    return scraped

# ── Run ───────────────────────────────────────────────────────────────────────
if not HTML_FILE.exists():
    print(f'⚠ File not found: {HTML_FILE.resolve()}')
    print('  Make sure mockshop_product.html is in the same folder as this notebook.')
else:
    reviews = scrape_mockshop(HTML_FILE)
    print(f'\n✓ Scraped {len(reviews)} reviews')
    print()
    for r in reviews[:3]:
        print(f'  [{r.rating}★] {r.author}: {r.text[:70]}...')


## Cell 6 — 👀 Preview Scraped Reviews

In [ ]:
if not reviews:
    print('⚠ No reviews — check Cell 5.')
else:
    raw_df = pd.DataFrame([{
        'author'   : r.author,
        'rating'   : r.rating,
        'date'     : r.date,
        'variation': r.variation,
        'helpful'  : r.helpful,
        'has_photo': r.has_photo,
        'text'     : r.text[:100] + '...' if len(r.text) > 100 else r.text,
    } for r in reviews])

    print(f'{len(raw_df)} reviews | Star distribution: {dict(raw_df["rating"].value_counts().sort_index())}')
    display(raw_df)


## Cell 7 — 🧠 Run Sentiment Analysis
Each review is scored by the VADER + TextBlob ensemble.  
Star rating contributes 20% weight to the final score.

In [ ]:
from collections import Counter

if not reviews:
    print('⚠ No reviews — run Cell 5 first.')
else:
    print(f'Analyzing {len(reviews)} reviews...\n')
    results = []
    for i, rev in enumerate(reviews):
        result = analyze(rev.text, rev.text, rev.rating)
        results.append(result)
        emoji = {'positive':'😊','negative':'😠','neutral':'😐','mixed':'🎭'}.get(result.sentiment, '')
        print(f'  [{rev.rating}★] {emoji} {result.sentiment:<10} ({result.confidence}% conf) — {rev.author}')

    dist  = Counter(r.sentiment for r in results)
    total = len(results)
    print()
    print('=' * 45)
    print('SENTIMENT BREAKDOWN')
    print('=' * 45)
    emojis = {'positive':'😊','negative':'😠','neutral':'😐','mixed':'🎭'}
    for s, n in dist.most_common():
        bar = '█' * int(n / total * 30)
        print(f'  {emojis.get(s,"")} {s:<10} {bar} {n} ({round(n/total*100)}%)')


## Cell 8 — 📊 Full Results Table

In [ ]:
if not results:
    print('⚠ Run Cell 7 first.')
else:
    results_df = pd.DataFrame([{
        'author'          : reviews[i].author,
        'star_rating'     : r.star_rating,
        'sentiment'       : r.sentiment,
        'confidence'      : r.confidence,
        'ensemble_score'  : round(r.ensemble_score, 3),
        'vader_compound'  : round(r.vader_compound, 3),
        'textblob_polarity': round(r.textblob_polarity, 3),
        'urgency'         : r.urgency,
        'frustration'     : r.frustration,
        'satisfaction'    : r.satisfaction,
        'loyalty_risk'    : r.loyalty_risk,
        'categories'      : ', '.join(r.categories),
        'intents'         : ', '.join(r.intents),
        'review_text'     : reviews[i].text[:80] + '...',
    } for i, r in enumerate(results)])

    print(f'Shape: {results_df.shape}')
    display(results_df)


## Cell 9 — 📈 Aggregate Statistics

In [ ]:
if not results:
    print('⚠ Run Cell 7 first.')
else:
    agg = aggregate(results)
    sep = '=' * 45
    print(sep)
    print('  AGGREGATE REPORT — MockShop Earbuds')
    print(sep)
    print(f'  Total reviews      : {agg["total_reviews"]}')
    print(f'  Dominant sentiment : {agg["dominant_sentiment"].upper()}')
    print(f'  Avg star rating    : {agg["avg_star_rating"]} / 5.0')
    print(f'  Avg confidence     : {agg["avg_confidence"]}%')
    print()
    print('  Sentiment distribution:')
    for s, n in agg['sentiment_distribution'].items():
        print(f'    {s:<12} : {n}')
    print()
    print('  Avg dimensions (0-100):')
    for k, v in agg['avg_dimensions'].items():
        bar = '█' * int(v // 5)
        print(f'    {k:<16} : {bar} {v}%')
    print()
    print(f'  ⚠  Churn risk      : {agg["churn_risk_reviews"]} reviews')
    print(f'  🔺 Escalation      : {agg["escalation_needed"]} reviews')
    kw = ", ".join(list(agg["top_keywords"].keys())[:8])
    print(f'  Top keywords       : {kw}')
    print(sep)


## Cell 10 — 📉 Visualization Charts

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
%matplotlib inline
plt.rcParams.update({
    'figure.facecolor' : '#ffffff',
    'axes.facecolor'   : '#fafafa',
    'font.family'      : 'monospace',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

if not results or not agg:
    print('⚠ Run Cells 7 and 9 first.')
else:
    COLORS = {'positive':'#1a7a42','negative':'#c0230e','neutral':'#4a6080','mixed':'#b07a00'}
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle('SentimentIQ — MockShop Product Review Analysis', fontsize=14, fontweight='bold', y=1.01)

    # 1. Sentiment donut
    ax = axes[0,0]
    d  = agg['sentiment_distribution']
    ax.pie(list(d.values()), labels=list(d.keys()), autopct='%1.0f%%',
           colors=[COLORS.get(k,'#999') for k in d],
           wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2), startangle=140)
    ax.set_title('Sentiment Distribution', fontweight='bold')

    # 2. Star rating bars
    ax = axes[0,1]
    star_counts = Counter(r.rating for r in reviews)
    counts = [star_counts.get(s, 0) for s in [1,2,3,4,5]]
    ax.bar(['1★','2★','3★','4★','5★'], counts,
           color=['#c0230e','#fb923c','#fbbf24','#86efac','#1a7a42'], edgecolor='white', width=0.6)
    ax.set_title('Star Rating Distribution', fontweight='bold')
    ax.set_ylabel('Reviews')
    for i, v in enumerate(counts):
        ax.text(i, v + 0.1, str(v), ha='center', fontsize=10)

    # 3. Top categories
    ax = axes[0,2]
    cats = dict(list(agg['top_categories'].items())[:6])
    ax.barh(list(cats.keys()), list(cats.values()), color='#ee4d2d', edgecolor='white')
    ax.set_title('Top Categories', fontweight='bold')
    ax.invert_yaxis()

    # 4. Dimensions
    ax = axes[1,0]
    dims = agg['avg_dimensions']
    dim_colors = ['#c0230e','#fb923c','#1a7a42','#6b3fa0']
    bars = ax.bar(list(dims.keys()), list(dims.values()), color=dim_colors, edgecolor='white')
    ax.set_ylim(0, 100)
    ax.set_title('Avg Sentiment Dimensions', fontweight='bold')
    ax.set_ylabel('Score (0-100)')
    for bar, v in zip(bars, dims.values()):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{v:.0f}%', ha='center', fontsize=9)

    # 5. Sentiment per star rating (stacked)
    ax = axes[1,1]
    star_sentiment = {s: Counter() for s in [1,2,3,4,5]}
    for rev, res in zip(reviews, results):
        star_sentiment[rev.rating][res.sentiment] += 1
    sentiments = ['positive','negative','neutral','mixed']
    x = [1,2,3,4,5]
    bottom = [0]*5
    for s in sentiments:
        vals = [star_sentiment[star].get(s,0) for star in x]
        ax.bar([str(i)+'★' for i in x], vals, bottom=bottom,
               label=s, color=COLORS.get(s,'#999'), edgecolor='white')
        bottom = [b+v for b,v in zip(bottom, vals)]
    ax.set_title('Sentiment per Star Rating', fontweight='bold')
    ax.set_ylabel('Reviews')
    ax.legend(fontsize=9)

    # 6. Confidence distribution
    ax = axes[1,2]
    for s, color in COLORS.items():
        confs = [r.confidence for r in results if r.sentiment == s]
        if confs:
            ax.hist(confs, bins=8, alpha=0.65, color=color, label=s, edgecolor='white')
    ax.set_title('Confidence Score Distribution', fontweight='bold')
    ax.set_xlabel('Confidence %')
    ax.set_ylabel('Reviews')
    ax.legend(fontsize=9)

    plt.tight_layout()
    chart_path = OUTPUT_DIR / f'charts_{TIMESTAMP}.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Chart saved → {chart_path}')


## Cell 11 — 🔎 Filter Reviews by Sentiment
Change `FILTER_BY` to explore any group.

In [ ]:
FILTER_BY = 'negative'  # 'positive' | 'negative' | 'neutral' | 'mixed'

if not results:
    print('⚠ Run Cell 7 first.')
else:
    filtered = [(rev, res) for rev, res in zip(reviews, results)
                if res.sentiment == FILTER_BY]
    print(f'{len(filtered)} {FILTER_BY.upper()} reviews:\n')
    sep = '-' * 60
    for i, (rev, res) in enumerate(filtered, 1):
        print(f'{i}. {rev.author} [{rev.rating}★] [{rev.date}] — {res.confidence}% confidence')
        print(f'   {rev.text}')
        print(f'   Categories : {res.categories}')
        print(f'   Intents    : {res.intents}')
        print(f'   Urgency={res.urgency}%  Frustration={res.frustration}%  Loyalty Risk={res.loyalty_risk}%')
        print(sep)


## Cell 12 — ⚠️ Churn Risk & Escalation Alerts

In [ ]:
if not results:
    print('⚠ Run Cell 7 first.')
else:
    churn = [(rev, res) for rev, res in zip(reviews, results) if 'Churn Risk' in res.intents]
    esc   = [(rev, res) for rev, res in zip(reviews, results) if 'Escalation Required' in res.intents]
    sep   = '=' * 60

    print(sep)
    print(f'  CHURN RISK — {len(churn)} review(s)')
    print(sep)
    for rev, res in churn:
        print(f'  [{rev.rating}★] {rev.author} ({rev.date})')
        print(f'  {rev.text}')
        print()

    print(sep)
    print(f'  ESCALATION REQUIRED — {len(esc)} review(s)')
    print(sep)
    for rev, res in esc:
        print(f'  [{rev.rating}★] {rev.author} ({rev.date})')
        print(f'  {rev.text}')
        print()


## Cell 13 — 💾 Save Reports
Generates three output files in `mockshop_output/`:
- **CSV** — open in Excel / Google Sheets
- **JSON** — full structured data + aggregate stats
- **HTML** — interactive dashboard, open in any browser

In [ ]:
if not results:
    print('⚠ Run Cell 7 first.')
else:
    csv_path  = save_csv(results,  OUTPUT_DIR / f'reviews_{TIMESTAMP}.csv')
    json_path = save_json(results, agg, OUTPUT_DIR / f'reviews_{TIMESTAMP}.json')
    html_path = save_html(results, agg, OUTPUT_DIR / f'report_{TIMESTAMP}.html',
                          product_url='mockshop_product.html')

    print(f'✓ CSV  → {csv_path}')
    print(f'✓ JSON → {json_path}')
    print(f'✓ HTML → {html_path}')
    print()
    print('Open the HTML file in your browser for the full interactive dashboard!')


## Cell 14 — 🖥️ Preview Dashboard Inside Notebook

In [ ]:
from IPython.display import IFrame, display
if html_path:
    display(IFrame(src=str(html_path), width='100%', height='860px'))
else:
    print('⚠ Run Cell 13 (Save) first.')
